# model-save-state-dict — ex2: atomic rank-0 save via tmp + os.replace + barrier

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `model-save-state-dict`. Running the final beacon cell reports progress against the `Distributed: model save state_dict rank-0` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: model save state_dict rank-0` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-save-state-dict`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-save-state-dict"
DD_SUBTOPIC = "Distributed: model save state_dict rank-0"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Atomic rank-0 save — tmp file + rename + barrier

Ex1's pattern was `if rank == 0: t.save(...); dist.barrier()`. Real training adds **atomicity** — a power loss or SIGKILL mid-`t.save` leaves a half-written file that fails to load on resume:

```python
if rank == 0:
    tmp = ckpt_path + '.tmp'
    t.save(model.state_dict(), tmp)
    os.replace(tmp, ckpt_path)   # POSIX atomic rename
dist.barrier()
```

**Why `os.replace`, not `os.rename`.** `os.replace` overwrites the destination if it exists — same semantics across POSIX and Windows. `os.rename` errors on Windows when the dest exists.

**Why the barrier still matters.** After rank 0 finishes the rename, the file is durable. But other ranks may already have charged ahead to the next step, and if subsequent code does `if rank == 1: load(ckpt)` they need to KNOW the write finished. `dist.barrier()` is the cheapest cross-rank fence — no data motion, just synchronization.

**Crash-resilience claim.** A crash AFTER `t.save(tmp)` but BEFORE `os.replace` leaves a stale `ckpt_path` and an orphaned `.tmp` file. The model can still resume from the previous good checkpoint. A crash DURING `t.save(tmp)` leaves a half-written `.tmp` file — but `ckpt_path` itself is untouched. The 'no half-written final file' guarantee is what atomic save buys you.

### Exercise 2 — atomic rank-0 save via tmp + os.replace + barrier

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `rank-0 save to tmp + os.replace + dist.barrier` pattern so the final checkpoint file never appears in a half-written state, verified by other ranks reading the file after the barrier.
> Keywords: state_dict, atomic-save, os.replace, barrier, checkpoint
> ```

**KCs targeted:** `rank0-tmp-then-rename`, `barrier-after-save-for-readers`

Implement `ex2_atomic_save(rank, world_size, dist_module, model, ckpt_path)`. Crash-resilient rank-0 save:

1. If `rank == 0`:
   a. Build `tmp_path = ckpt_path + '.tmp'`.
   b. `t.save(model.state_dict(), tmp_path)`.
   c. `os.replace(tmp_path, ckpt_path)` — POSIX atomic rename.
2. ALL ranks (including rank 0): call `dist_module.barrier()` — non-zero ranks block here until rank 0 finishes the rename.
3. Return `ckpt_path` (so the caller has the final path).

Guarantees the test verifies:
- After this function returns, `ckpt_path` exists, `ckpt_path + '.tmp'` does NOT exist.
- Loading from `ckpt_path` on any rank gives back the same state_dict that rank 0 saved.
- The barrier was actually called (so non-zero ranks didn't race ahead).

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `model` — `nn.Module`; `ckpt_path` — str.
Output: `str` — the final checkpoint path.

In [ ]:
def ex2_atomic_save(rank: int, world_size: int, dist_module, model: 'nn.Module', ckpt_path: str) -> str:
    import os
    if rank == 0:
        tmp_path = ckpt_path + '.tmp'
        t.save(model.state_dict(), tmp_path)
        os.replace(tmp_path, ckpt_path)
    dist_module.barrier()
    return ckpt_path


<details><summary>Solution</summary>

```python
def ex2_atomic_save(rank: int, world_size: int, dist_module, model: 'nn.Module', ckpt_path: str) -> str:
    import os
    if rank == 0:
        tmp_path = ckpt_path + '.tmp'
        t.save(model.state_dict(), tmp_path)
        os.replace(tmp_path, ckpt_path)
    dist_module.barrier()
    return ckpt_path
```

**`os.replace` vs `os.rename`.** `os.replace` overwrites the dest if it exists, on every OS. `os.rename` errors on Windows when the dest exists. For cross-platform code, always reach for `replace`.

**Why the barrier even though only rank 0 writes.** Non-zero ranks didn't write anything, so they have nothing to wait for in ISOLATION. The barrier is for the CALLER's benefit — once `ex2_atomic_save` returns on every rank, downstream code can assume the file is durable. If a downstream `if rank == 1: load(ckpt)` ran without a prior barrier, rank 1 might attempt the load before rank 0 has finished writing.

**Atomic write + barrier are independent guarantees.** Atomicity means 'never half-written on disk'. Barrier means 'every rank agrees the write is done'. Both are needed for crash-resilient multi-rank checkpointing.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()